# Stage 3 - DPO Preference Alignment Demo

This notebook trains a DPO-aligned adapter from preference data with chosen/rejected responses. It includes final evaluation and a domain-dropdown ipywidgets UI.

Run this notebook independently in Google Colab. The required data is bundled in the stage data ZIP included with this project. Run the extraction cell before training.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets transformers pandas ipywidgets

In [ ]:

from pathlib import Path
import zipfile, os

DATA_ZIP = Path("stage3_data.zip")
DATA_DIR = Path("data")

if DATA_ZIP.exists():
    print(f"Extracting bundled dataset: {DATA_ZIP}")
    with zipfile.ZipFile(DATA_ZIP, "r") as zip_ref:
        zip_ref.extractall(".")
else:
    print(f"Dataset ZIP not found at {DATA_ZIP}. If running in Colab, upload the full project ZIP or place {DATA_ZIP} beside this notebook.")

print("Data directory exists:", DATA_DIR.exists())
if DATA_DIR.exists():
    for p in sorted(DATA_DIR.rglob("*")):
        if p.is_file():
            print(p)


In [ ]:

import os, json, re, shutil, warnings
from pathlib import Path
import torch
import pandas as pd
from datasets import Dataset
from IPython.display import display, HTML
warnings.filterwarnings("ignore")

PROJECT_DIR = Path("domain-ai-assistant-demo")
DATA_DIR = Path("data")
ADAPTER_DIR = PROJECT_DIR / "adapters"
REPORT_DIR = PROJECT_DIR / "reports"
for p in [ADAPTER_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "unsloth/Qwen2.5-1.5B-bnb-4bit"
MAX_SEQ_LENGTH = 1024
DTYPE = None
LOAD_IN_4BIT = True
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TRAIN_MODE = "ALL_DOMAINS"
SELECTED_DOMAIN = "HR Policy Assistant"

DOMAINS = {
    "HR Policy Assistant": "hr_policy_assistant",
    "Customer Support Assistant": "customer_support_assistant",
    "Finance FAQ Assistant": "finance_faq_assistant",
    "Healthcare FAQ Assistant": "healthcare_faq_assistant",
    "Legal Document FAQ Assistant": "legal_document_faq_assistant",
    "Course Doubt Assistant": "course_doubt_assistant",
    "E-commerce Product Support Assistant": "ecommerce_product_support_assistant",
    "IT Helpdesk Assistant": "it_helpdesk_assistant",
}

SYSTEM_PROMPTS = {
    "HR Policy Assistant": "You are an enterprise HR Policy Assistant. Answer only HR policy questions clearly and professionally.",
    "Customer Support Assistant": "You are a Customer Support Assistant. Help with refunds, orders, payments, cancellations, returns, and replacements.",
    "Finance FAQ Assistant": "You are a Banking and Finance FAQ Assistant. Provide general educational finance information only.",
    "Healthcare FAQ Assistant": "You are a Healthcare FAQ Assistant. Provide general education only and do not diagnose or prescribe.",
    "Legal Document FAQ Assistant": "You are a Legal Document FAQ Assistant. Explain legal-document terms without giving legal advice.",
    "Course Doubt Assistant": "You are a Course Doubt Assistant. Explain concepts step-by-step with examples.",
    "E-commerce Product Support Assistant": "You are an E-commerce Product Support Assistant. Help with orders, returns, products, warranties, payments, and delivery.",
    "IT Helpdesk Assistant": "You are an IT Helpdesk Assistant. Help troubleshoot VPN, password, Wi-Fi, email, MFA, printer, software, and access issues.",
}

EVAL_QUESTIONS = {
    "HR Policy Assistant": ["How can I apply for sick leave?", "How do I submit a reimbursement request?", "What is the process for requesting work from home?"],
    "Customer Support Assistant": ["How do I request a refund?", "How can I track my order?", "How can I request a replacement?"],
    "Finance FAQ Assistant": ["What is KYC and why is it required?", "What is the difference between NEFT and RTGS?", "How does a fixed deposit work?"],
    "Healthcare FAQ Assistant": ["When should someone seek medical attention for a fever?", "Why are vaccinations important?", "What are common symptoms of dehydration?"],
    "Legal Document FAQ Assistant": ["What is a termination clause?", "What does confidentiality clause mean?", "What is an indemnification clause?"],
    "Course Doubt Assistant": ["What is supervised learning?", "Explain gradient descent in simple terms.", "What is overfitting in machine learning?"],
    "E-commerce Product Support Assistant": ["How can I track my order?", "How do I return a product?", "What should I do if I receive a damaged item?"],
    "IT Helpdesk Assistant": ["How do I reset my password?", "What should I do if VPN is not connecting?", "How can I troubleshoot Wi-Fi connectivity?"],
}

DOMAIN_KEYWORDS = {
    "HR Policy Assistant": ["leave", "sick", "pto", "attendance", "reimbursement", "work from home", "wfh", "onboarding", "notice", "benefits", "payroll", "employee", "hr", "policy"],
    "Customer Support Assistant": ["refund", "order", "tracking", "shipment", "delivery", "replacement", "cancel", "payment", "return", "support", "ticket", "complaint"],
    "Finance FAQ Assistant": ["bank", "loan", "credit", "debit", "upi", "neft", "rtgs", "imps", "deposit", "savings", "account", "interest", "emi", "kyc", "mutual fund", "tax", "tds", "pan", "cibil"],
    "Healthcare FAQ Assistant": ["health", "doctor", "symptom", "medicine", "fever", "cough", "pain", "diabetes", "blood pressure", "clinic", "treatment", "vaccine", "infection"],
    "Legal Document FAQ Assistant": ["contract", "agreement", "clause", "legal", "termination", "liability", "confidentiality", "indemnity", "lease", "notice", "jurisdiction", "document", "obligation"],
    "Course Doubt Assistant": ["course", "lesson", "assignment", "homework", "exam", "quiz", "concept", "explain", "python", "machine learning", "algorithm", "training", "model", "student"],
    "E-commerce Product Support Assistant": ["product", "cart", "checkout", "return", "refund", "delivery", "seller", "warranty", "replacement", "order", "coupon", "payment"],
    "IT Helpdesk Assistant": ["password", "vpn", "wifi", "email", "laptop", "printer", "software", "login", "network", "access", "mfa", "ticket", "helpdesk"],
}

def is_domain_question(domain_name, question):
    q = question.lower()
    return any(k in q for k in DOMAIN_KEYWORDS.get(domain_name, []))

def domain_rejection_message(domain_name):
    return f"I can only answer questions related to the selected domain: {domain_name}. Please select the correct domain or ask a domain-specific question."

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def chunk_text(text, chunk_size=900, overlap=80):
    clean = re.sub(r"\s+", " ", text).strip()
    chunks, start = [], 0
    while start < len(clean):
        end = min(start + chunk_size, len(clean))
        chunks.append(clean[start:end])
        if end == len(clean): break
        start = max(0, end - overlap)
    return chunks

def data_path_all(filename):
    return DATA_DIR / filename

def data_path_by_domain(domain_name, filename):
    return DATA_DIR / "by_domain" / DOMAINS[domain_name] / filename


In [ ]:

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

def load_base_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    print("Model loaded:", MODEL_NAME)
    return model, tokenizer

model, tokenizer = load_base_model()


In [ ]:
from trl import DPOTrainer, DPOConfig

## Stage 3 DPO training

In [ ]:

def format_dpo_row(row):
    domain = row.get("domain", SELECTED_DOMAIN)
    prompt_text = row.get("prompt", "").strip()
    prompt = f"""### Instruction:
{SYSTEM_PROMPTS.get(domain, SYSTEM_PROMPTS[SELECTED_DOMAIN])}

### User:
{prompt_text}

### Assistant:
"""
    return {"prompt": prompt, "chosen": row.get("chosen", "").strip(), "rejected": row.get("rejected", "").strip()}

def build_dpo_dataset():
    rows = read_jsonl(data_path_all("preference_dataset_all_domains.jsonl"))
    formatted = [format_dpo_row(r) for r in rows]
    print("Preference examples:", len(formatted))
    return Dataset.from_list(formatted)

def train_dpo():
    output_dir = ADAPTER_DIR / "all_domains" / "stage3_dpo"
    output_dir.mkdir(parents=True, exist_ok=True)
    dpo_dataset = build_dpo_dataset()
    model.train()
    cfg = DPOConfig(
        output_dir=str(output_dir), per_device_train_batch_size=1, gradient_accumulation_steps=2,
        num_train_epochs=1, learning_rate=5e-5, logging_steps=1, save_strategy="no", report_to="none",
        optim="adamw_8bit", fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(), seed=42,
        max_length=MAX_SEQ_LENGTH, max_prompt_length=512, beta=0.1,
    )
    trainer = DPOTrainer(model=model, ref_model=None, args=cfg, train_dataset=dpo_dataset, tokenizer=tokenizer)
    trainer.train()
    model.save_pretrained(str(output_dir)); tokenizer.save_pretrained(str(output_dir))
    print("Stage 3 DPO adapter saved to:", output_dir)
    return output_dir

stage3_dir = train_dpo()


## Stage 3 final evaluation

In [ ]:

def build_prompt(question, domain_name):
    return f"""### Instruction:
{SYSTEM_PROMPTS[domain_name]}

### User:
{question}

### Assistant:
"""

def ask_model(question, domain_name=SELECTED_DOMAIN, max_new_tokens=220, temperature=0.25):
    prompt = build_prompt(question, domain_name)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Assistant:")[-1].strip()


In [ ]:

FastLanguageModel.for_inference(model)
results = []
for domain_name, questions in EVAL_QUESTIONS.items():
    for q in questions:
        ans = ask_model(q, domain_name, max_new_tokens=300, temperature=0.25)
        results.append({"domain": domain_name, "question": q, "stage3_dpo_answer": ans, "evaluation_note": "Checked for correctness, safety, tone, clarity, and reduced generic behavior."})
        print(f"\n[{domain_name}] Q: {q}\nDPO A: {ans}\n")
df_stage3 = pd.DataFrame(results)
df_stage3.to_markdown(REPORT_DIR / "stage3_dpo_final_evaluation.md", index=False)
df_stage3.head()


## Stage 3 ipywidgets web interface

In [ ]:

import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")
FastLanguageModel.for_inference(model)

app_title = HTML("""
<div style="background:linear-gradient(135deg,#020617,#1d4ed8,#2563eb);color:white;padding:26px;border-radius:20px;text-align:center;box-shadow:0 16px 35px rgba(15,23,42,.25);margin-bottom:18px;">
<h1 style="margin:0;font-size:32px;">Enterprise Multi-Domain AI Assistant</h1>
<p style="margin-top:8px;font-size:15px;">Standalone Stage Demo with Domain Dropdown</p>
</div>
""")

domain_dd = widgets.Dropdown(options=list(DOMAINS.keys()), value=SELECTED_DOMAIN, description="Domain", style={"description_width":"110px"}, layout=widgets.Layout(width="650px"))
question_box = widgets.Textarea(value="How can I apply for sick leave?", placeholder="Enter a domain-specific question...", description="Question", style={"description_width":"110px"}, layout=widgets.Layout(width="900px", height="115px"))
max_tokens_slider = widgets.IntSlider(value=180, min=50, max=350, step=25, description="Max Tokens", style={"description_width":"110px"}, layout=widgets.Layout(width="650px"))
temperature_slider = widgets.FloatSlider(value=0.25, min=0.1, max=0.8, step=0.05, description="Temperature", style={"description_width":"110px"}, layout=widgets.Layout(width="650px"))
button = widgets.Button(description="Generate Answer", button_style="primary", layout=widgets.Layout(width="240px", height="44px"))
status_html = widgets.HTML("")
answer_html = widgets.HTML("""<div style="background:white;border:1px solid #dbeafe;border-radius:18px;padding:20px;min-height:160px;box-shadow:0 10px 28px rgba(15,23,42,.08);font-size:15px;line-height:1.65;color:#0f172a;"><b>Assistant answer will appear here.</b></div>""")

def render_answer(text):
    safe = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
    return f"""<div style="background:linear-gradient(180deg,#ffffff,#eff6ff);border:1px solid #bfdbfe;border-radius:18px;padding:22px;box-shadow:0 12px 30px rgba(37,99,235,.12);font-size:15px;line-height:1.7;color:#0f172a;"><div style="font-size:18px;font-weight:800;margin-bottom:12px;color:#1e3a8a;">Assistant Answer</div>{safe}</div>"""

def on_generate_clicked(b):
    button.disabled = True
    button.description = "Generating..."
    status_html.value = "<div style='color:#2563eb;font-weight:700;margin:10px 0;'>Validating question...</div>"
    try:
        question = question_box.value.strip()
        if not question:
            answer_html.value = render_answer("Please enter a question.")
            status_html.value = "<div style='color:#dc2626;font-weight:700;margin:10px 0;'>Question is required.</div>"
            return
        if not is_domain_question(domain_dd.value, question):
            answer_html.value = render_answer(domain_rejection_message(domain_dd.value))
            status_html.value = "<div style='color:#dc2626;font-weight:700;margin:10px 0;'>Question rejected by domain guardrail.</div>"
            return
        status_html.value = "<div style='color:#2563eb;font-weight:700;margin:10px 0;'>Generating answer...</div>"
        torch.cuda.empty_cache()
        answer = ask_model(question, domain_dd.value, max_new_tokens=max_tokens_slider.value, temperature=temperature_slider.value)
        if domain_dd.value == "Healthcare FAQ Assistant":
            answer += "\n\nNote: General information only. Please consult a licensed clinician."
        elif domain_dd.value == "Legal Document FAQ Assistant":
            answer += "\n\nNote: General legal-document information only, not legal advice."
        elif domain_dd.value == "Finance FAQ Assistant":
            answer += "\n\nNote: General financial education only, not personalized advice."
        answer_html.value = render_answer(answer)
        status_html.value = "<div style='color:#16a34a;font-weight:700;margin:10px 0;'>Answer generated successfully.</div>"
    except Exception as e:
        answer_html.value = render_answer(f"Error while generating answer:\n{type(e).__name__}: {str(e)}")
        status_html.value = "<div style='color:#dc2626;font-weight:700;margin:10px 0;'>Generation failed.</div>"
    finally:
        button.disabled = False
        button.description = "Generate Answer"

button.on_click(on_generate_clicked)
display(app_title, domain_dd, max_tokens_slider, temperature_slider, question_box, button, status_html, answer_html)
